## Experience 3 : detection et segmentation d'instances
Comparaison de **YOLO11-seg**, **YOLO26-seg** et **Mask R-CNN** sur `small_dataset` (polygones YOLO, 3 classes).

**Ce notebook est autonome** : les constantes (`CLASS_NAMES`, `IMAGE_EXTENSIONS`, ...) sont definies dans la premiere cellule, il n'est plus necessaire d'avoir execute le notebook de classification avant.

**Dependances** : `%pip install -U ultralytics torch torchvision pyyaml pillow` (YOLO26 exige une version recente d'Ultralytics). Avec PyTorch 2.2.x, garder `numpy<2`.

**Materiel.** Execution sur **CPU par defaut** (`DEVICE_PREFERENCE = "cpu"`) : le noyau plantait sur MPS (Mac) dans l'experience de classification. L'entrainement CPU est lent (YOLO : 50 epoques, Mask R-CNN : 10 epoques). Pour utiliser le GPU, mettre `"auto"` ou `"mps"` dans la premiere cellule.

**Execution.** Executer les cellules dans l'ordre; chaque entrainement a sa propre cellule (un plantage ne fait pas perdre les resultats precedents). Les resultats de test utilisent la **meilleure epoque** (choisie sur valid), pas la derniere.

In [1]:
# EXPERIENCE 3 : DETECTION ET SEGMENTATION D'INSTANCES - CONFIGURATION
# Comparaison : YOLO11-seg vs YOLO26-seg vs Mask R-CNN.
# Dependances : %pip install -U ultralytics torch torchvision pyyaml pillow
# YOLO sauvegarde ses traces d'entrainement dans outputs/detection/train et
# ses resultats d'evaluation dans outputs/detection/evaluation.

import os

# Doit etre defini AVANT l'import de torch : repli sur le CPU pour les operateurs absents de MPS.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import copy
import csv
import gc
import random
from pathlib import Path

import torch
import yaml
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights, maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from ultralytics import YOLO

# Constantes (auparavant definies dans le notebook de classification : le notebook est autonome).
CLASS_NAMES = ["glioma", "meningioma", "pituitary"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42
PRETRAINED = True  # False = Mask R-CNN aleatoire (utile hors ligne, sans telechargement des poids)

# "cpu" par defaut (le noyau plantait sur MPS dans l'experience de classification).
# Passer a "auto", "mps" ou "cuda" pour utiliser l'accelerateur. L'entrainement CPU est lent.
DEVICE_PREFERENCE = "cpu"

# YOLO valide sur valid a chaque epoque; ensuite on evalue une fois sur test.
YOLO_EPOCHS = 50
YOLO_IMAGE_SIZE = 640
YOLO_BATCH_SIZE = 8
MASK_RCNN_EPOCHS = 10
MASK_RCNN_BATCH_SIZE = 2
# Taille interne de Mask R-CNN (torchvision : 800 par defaut). 512 reduit memoire et temps ;
# les masques predits sont rescales a la taille d'origine avant l'evaluation.
MASK_RCNN_MIN_SIZE = 512

DATASET_ROOT = Path("small_dataset").resolve()
detection_train_root = Path("outputs/detection/train").resolve()
detection_eval_root = Path("outputs/detection/evaluation").resolve()
detection_train_root.mkdir(parents=True, exist_ok=True)
detection_eval_root.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)


def select_device(preference):
    if preference == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    if preference == "mps" and torch.backends.mps.is_available():
        return torch.device("mps")
    if preference == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
    return torch.device("cpu")


def free_device_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


set_seed(SEED)
DEVICE = select_device(DEVICE_PREFERENCE)
# Ultralytics attend "cpu", "mps" ou un indice de GPU CUDA.
YOLO_DEVICE = 0 if DEVICE.type == "cuda" else DEVICE.type
print(f"PyTorch {torch.__version__} | materiel : {DEVICE} | graine={SEED}")

# Verifier le dataset et generer le YAML YOLO (chemins absolus vers les splits existants).
for split in ("train", "valid", "test"):
    for sub_dir in ("images", "labels"):
        if not (DATASET_ROOT / split / sub_dir).is_dir():
            raise FileNotFoundError(f"Dossier introuvable : {DATASET_ROOT / split / sub_dir}")

detection_yaml = detection_train_root / "data.yaml"
yaml_content = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {index: name for index, name in enumerate(CLASS_NAMES)},
}
detection_yaml.write_text(yaml.safe_dump(yaml_content, sort_keys=False), encoding="utf-8")
print(f"YAML YOLO : {detection_yaml}")

PyTorch 2.2.2 | materiel : cpu | graine=42
YAML YOLO : /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/outputs/detection/train/data.yaml


In [2]:
# YOLO : entrainement puis evaluation sur test avec les MEILLEURS poids (best.pt).
yolo_models = {}
yolo_train_dirs = {}
yolo_test_results = []


def train_and_evaluate_yolo(model_name, weights_path):
    model_key = model_name.lower().replace("-", "_")
    yolo_model = YOLO(weights_path)
    yolo_model.train(
        data=str(detection_yaml),
        epochs=YOLO_EPOCHS,
        imgsz=YOLO_IMAGE_SIZE,
        batch=YOLO_BATCH_SIZE,
        patience=10,
        device=YOLO_DEVICE,
        seed=SEED,
        project=str(detection_train_root),  # chemin absolu : pas de prefixe runs/segment/
        name=model_key,
        exist_ok=True,  # chemins stables : une relance ecrase au lieu de creer yolo11_seg2, ...
        plots=True,
    )
    train_dir = Path(yolo_model.trainer.save_dir)
    best_weights = train_dir / "weights" / "best.pt"
    if not best_weights.is_file():
        raise FileNotFoundError(f"best.pt introuvable pour {model_name} dans {train_dir}")

    # Recharger explicitement best.pt pour l'evaluation finale sur test.
    best_model = YOLO(str(best_weights))
    yolo_metrics = best_model.val(
        data=str(detection_yaml),
        split="test",
        imgsz=YOLO_IMAGE_SIZE,
        batch=YOLO_BATCH_SIZE,
        device=YOLO_DEVICE,
        project=str(detection_eval_root),
        name=model_key,
        exist_ok=True,
        plots=True,
    )
    precision = float(yolo_metrics.seg.mp)  # precision moyenne (masques, IoU 0.5, seuil de confiance optimal)
    recall = float(yolo_metrics.seg.mr)
    result = {
        "model": model_name,
        "box_map50": float(yolo_metrics.box.map50),
        "box_map50_95": float(yolo_metrics.box.map),
        "mask_map50": float(yolo_metrics.seg.map50),
        "mask_map50_95": float(yolo_metrics.seg.map),
        "precision_iou50": precision,
        "recall_iou50": recall,
        "f1_iou50": 2 * precision * recall / max(precision + recall, 1e-12),
    }
    print(
        f"{model_name} | masque mAP50={result['mask_map50']:.4f} | "
        f"masque mAP50-95={result['mask_map50_95']:.4f}"
    )
    yolo_models[model_name] = best_model
    yolo_train_dirs[model_name] = train_dir
    yolo_test_results.append(result)
    free_device_memory()
    return result

In [3]:
# ENTRAINEMENT YOLO 1/2 : YOLO11-seg
train_and_evaluate_yolo("YOLO11-seg", "yolo11n-seg.pt")

Ultralytics 8.4.173 🚀 Python-3.11.16 torch-2.2.2 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/outputs/detection/train/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11_

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


YOLO11n-seg summary: 203 layers, 2,843,193 parameters, 2,843,177 gradients, 9.8 GFLOPs

Transferred 510/561 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 93.1±7.0 MB/s, size: 32.6 KB)
train: Scanning /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/small_dataset/train/labels... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 1.7Kit/s 0.2s<0.3s
train: New cache created: /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/small_dataset/train/labels.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 114.0±32.2 MB/s, size: 30.6 KB)
val: Scanning /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/small_dataset/valid/labels... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 2.0Kit/s 0.1s
val: New cache created: /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/small_dataset/valid/labels.cache
optimizer: 'optimizer=auto' found

/Users/ikram/miniconda3/envs/mon_env_etudeCas/lib/python3.11/site-packages/polars/_cpu_check.py:271: RuntimeWarning: Missing required CPU features.

The following required CPU features were not detected:
    avx, avx2, fma, bmi1, bmi2, lzcnt, movbe
Continuing to use this version of Polars on this processor will likely result in a crash.
Install `polars[rtcompat]` instead of `polars` to run Polars with better compatibility.

Hint: If you are on an Apple ARM machine (e.g. M1) this is likely due to running Python under Rosetta.
It is recommended to install a native version of Python that does not run under Rosetta x86-64 emulation.

If you believe this warning to be a false positive, you can set the `POLARS_SKIP_CPU_CHECK` environment variable to bypass this check.

  warnings.warn(


Plotting labels to /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/outputs/detection/train/yolo11_seg/labels.jpg... 
Using 300 train, 150 val images for fraction=1.0 at imgsz=640
Using 0 dataloader workers
Logging results to /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/outputs/detection/train/yolo11_seg
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       1/50         0G       1.25      2.573      4.116      1.339          0          8        640: 100% ━━━━━━━━━━━━ 38/38 8.3s/it 5:176.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.9s/it 58.7s.5ss
                   all        150        150    0.00374        0.8      0.215      0.144     0.0037       0.78      0.219      0.142

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   se

{'model': 'YOLO11-seg',
 'box_map50': 0.8186845897155196,
 'box_map50_95': 0.6165610634150075,
 'mask_map50': 0.8213974432702003,
 'mask_map50_95': 0.612930064754605,
 'precision_iou50': 0.7792856488002394,
 'recall_iou50': 0.824863854531935,
 'f1_iou50': 0.8014272519058074}

In [4]:
# ENTRAINEMENT YOLO 2/2 : YOLO26-seg (necessite une version recente : %pip install -U ultralytics)
train_and_evaluate_yolo("YOLO26-seg", "yolo26n-seg.pt")

New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.11.16 torch-2.2.2 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/outputs/detection/train/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, 

{'model': 'YOLO26-seg',
 'box_map50': 0.794643376882389,
 'box_map50_95': 0.5767424598438572,
 'mask_map50': 0.8019022080370144,
 'mask_map50_95': 0.5952166405446193,
 'precision_iou50': 0.8211501753804128,
 'recall_iou50': 0.74069786331416,
 'f1_iou50': 0.7788519309121607}

In [5]:
# MASK R-CNN : convertir les polygones YOLO en masques, boites et classes.
def pil_to_float_tensor(image):
    """Image PIL RGB -> tenseur float (3, H, W) dans [0, 1], sans passer par NumPy."""
    width, height = image.size
    pixels = torch.frombuffer(bytearray(image.tobytes()), dtype=torch.uint8)
    return pixels.view(height, width, 3).permute(2, 0, 1).float() / 255.0


class YoloInstanceDataset(Dataset):
    def __init__(self, split):
        self.images_dir = DATASET_ROOT / split / "images"
        self.labels_dir = DATASET_ROOT / split / "labels"
        self.image_paths = [
            image_path
            for image_path in sorted(self.images_dir.iterdir())
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS
        ]
        print(f"{split}: {len(self.image_paths)} images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        label_path = self.labels_dir / f"{image_path.stem}.txt"
        with Image.open(image_path) as image:  # fermer le fichier apres lecture
            image = image.convert("RGB")
            width, height = image.size
            image_tensor = pil_to_float_tensor(image)

        instance_masks = []
        boxes = []
        labels = []
        if label_path.is_file():
            with label_path.open("r") as label_file:
                for line in label_file:
                    fields = line.split()
                    if len(fields) < 7:
                        continue
                    try:
                        class_id = int(fields[0])
                        coordinates = [float(value) for value in fields[1:]]
                    except ValueError:
                        continue
                    if not 0 <= class_id < len(CLASS_NAMES) or len(coordinates) % 2:
                        continue

                    points = [
                        (
                            round(min(max(coordinates[i], 0.0), 1.0) * (width - 1)),
                            round(min(max(coordinates[i + 1], 0.0), 1.0) * (height - 1)),
                        )
                        for i in range(0, len(coordinates), 2)
                    ]
                    if len(points) < 3:
                        continue

                    mask_image = Image.new("L", (width, height), 0)
                    ImageDraw.Draw(mask_image).polygon(points, fill=1)
                    bounds = mask_image.getbbox()
                    if bounds is None:
                        continue
                    left, top, right, bottom = bounds
                    if right <= left or bottom <= top:
                        continue
                    mask = torch.frombuffer(bytearray(mask_image.tobytes()), dtype=torch.uint8)
                    instance_masks.append(mask.view(height, width))
                    boxes.append([left, top, right, bottom])
                    labels.append(class_id + 1)  # Zero est reserve a l'arriere-plan.

        if instance_masks:
            masks_tensor = torch.stack(instance_masks)
        else:
            masks_tensor = torch.zeros((0, height, width), dtype=torch.uint8)
        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": masks_tensor,
            "image_id": torch.tensor([index]),
            "area": masks_tensor.flatten(1).sum(dim=1).to(torch.float32),
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64),
        }
        return image_tensor, target


def detection_collate_fn(batch):
    """Conserver les listes d'images et de cibles de tailles variables."""
    return tuple(zip(*batch))


# Conserver les jeux train, validation et test dans leurs roles respectifs.
detection_datasets = {split: YoloInstanceDataset(split) for split in ("train", "valid", "test")}
if any(len(dataset) == 0 for dataset in detection_datasets.values()):
    raise ValueError("Au moins un split ne contient aucune image.")
detection_loaders = {tr
    split: DataLoader(
        dataset,
        batch_size=MASK_RCNN_BATCH_SIZE,
        shuffle=(split == "train"),
        collate_fn=detection_collate_fn,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED) if split == "train" else None,
    )
    for split, dataset in detection_datasets.items()
}

train: 300 images
valid: 150 images
test: 150 images


In [6]:
# MASK R-CNN : modele, evaluation et boucle d'entrainement.
def build_mask_rcnn():
    """Mask R-CNN avec poids COCO et tetes adaptees aux trois classes (+ arriere-plan)."""
    try:
        model = maskrcnn_resnet50_fpn(
            weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT if PRETRAINED else None,
            weights_backbone=None,
            min_size=MASK_RCNN_MIN_SIZE,
            max_size=MASK_RCNN_MIN_SIZE * 2,
        )
    except Exception as error:  # telechargement impossible (hors ligne, proxy...)
        raise RuntimeError(
            "Impossible de charger les poids COCO de Mask R-CNN. Verifier la connexion ou "
            "mettre PRETRAINED = False (la comparaison perd alors son sens)."
        ) from error
    number_of_classes = len(CLASS_NAMES) + 1
    box_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, number_of_classes)
    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, number_of_classes)
    return model


# Precision, rappel et F1 des masques avec un IoU minimal de 0.5 (calcul avec torch, sans NumPy).
@torch.no_grad()
def evaluate_mask_rcnn(model, data_loader, confidence_threshold=0.5, iou_threshold=0.5):
    model.eval()
    true_positives = 0
    false_positives = 0
    false_negatives = 0

    for images, targets in data_loader:
        predictions = model([image.to(DEVICE) for image in images])
        for prediction, target in zip(predictions, targets):
            scores = prediction["scores"].cpu()
            predicted_labels = prediction["labels"].cpu()
            predicted_masks = prediction["masks"].cpu()[:, 0] >= 0.5
            ground_truth_labels = target["labels"]
            ground_truth_masks = target["masks"].bool()
            matched_ground_truth = set()

            # Associer chaque prediction (par score decroissant) a une instance de meme classe.
            for prediction_index in torch.argsort(scores, descending=True).tolist():
                if scores[prediction_index] < confidence_threshold:
                    break  # scores tries : toutes les suivantes sont sous le seuil
                best_index = None
                best_iou = 0.0
                for ground_truth_index in range(len(ground_truth_labels)):
                    if (
                        ground_truth_labels[ground_truth_index] != predicted_labels[prediction_index]
                        or ground_truth_index in matched_ground_truth
                    ):
                        continue
                    predicted_mask = predicted_masks[prediction_index]
                    target_mask = ground_truth_masks[ground_truth_index]
                    intersection = (predicted_mask & target_mask).sum().item()
                    union = (predicted_mask | target_mask).sum().item()
                    current_iou = intersection / union if union else 0.0
                    if current_iou > best_iou:
                        best_iou = current_iou
                        best_index = ground_truth_index
                if best_index is not None and best_iou >= iou_threshold:
                    true_positives += 1
                    matched_ground_truth.add(best_index)
                else:
                    false_positives += 1
            false_negatives += len(ground_truth_labels) - len(matched_ground_truth)

    precision = true_positives / max(true_positives + false_positives, 1)
    recall = true_positives / max(true_positives + false_negatives, 1)
    f1_score = 2 * precision * recall / max(precision + recall, 1e-12)
    return precision, recall, f1_score


def write_history_csv(history, path):
    with Path(path).open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)


mask_rcnn_train_dir = detection_train_root / "mask_rcnn"
mask_rcnn_train_dir.mkdir(parents=True, exist_ok=True)


# Entrainement avec validation apres chaque epoque; le test utilise la meilleure epoque (F1 valid).
def train_mask_rcnn():
    set_seed(SEED)
    model = build_mask_rcnn().to(DEVICE)
    optimizer = torch.optim.SGD(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=0.005,
        momentum=0.9,
        weight_decay=0.0005,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    history = []
    best_key = (-1.0, 0.0)  # (F1 valid, -perte train) : departage les egalites
    best_state = None
    best_epoch = 0

    for epoch in range(MASK_RCNN_EPOCHS):
        model.train()
        total_loss = 0.0
        for images, targets in detection_loaders["train"]:
            images = [image.to(DEVICE) for image in images]
            targets = [{key: value.to(DEVICE) for key, value in target.items()} for target in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_value for loss_value in loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        train_loss = total_loss / len(detection_loaders["train"])
        valid_precision, valid_recall, valid_f1 = evaluate_mask_rcnn(model, detection_loaders["valid"])
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "valid_precision": valid_precision,
                "valid_recall": valid_recall,
                "valid_f1_iou50": valid_f1,
            }
        )
        print(
            f"Mask R-CNN | epoque {epoch + 1}/{MASK_RCNN_EPOCHS} | "
            f"perte train {train_loss:.4f} | F1 valid {valid_f1:.4f}"
        )

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if (valid_f1, -train_loss) > best_key:
            best_key, best_state, best_epoch = (valid_f1, -train_loss), state, epoch + 1
            torch.save(
                {"epoch": best_epoch, "model_state_dict": best_state, "class_names": CLASS_NAMES},
                mask_rcnn_train_dir / "mask_rcnn_best.pth",
            )
        # Dernier etat pour reprendre apres un arret du noyau.
        torch.save(
            {"epoch": epoch + 1, "model_state_dict": state, "history": history, "class_names": CLASS_NAMES},
            mask_rcnn_train_dir / "mask_rcnn_latest.pth",
        )
        write_history_csv(history, mask_rcnn_train_dir / "training_history.csv")

    # Evaluation finale sur test avec la meilleure epoque (choisie sur valid, jamais sur test).
    model.load_state_dict(best_state)
    precision, recall, f1_score = evaluate_mask_rcnn(model, detection_loaders["test"])
    print(
        f"Mask R-CNN test (meilleure epoque {best_epoch}) | precision={precision:.4f} | "
        f"rappel={recall:.4f} | F1 masque={f1_score:.4f} (IoU >= 0.5)"
    )
    model = model.cpu()
    del optimizer, scheduler
    free_device_memory()
    return model, history, {
        "best_epoch": best_epoch,
        "precision_iou50": precision,
        "recall_iou50": recall,
        "f1_iou50": f1_score,
    }

In [7]:
# ENTRAINEMENT MASK R-CNN
mask_rcnn_model, mask_rcnn_history, mask_rcnn_test = train_mask_rcnn()

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /Users/ikram/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth
100.0%


Mask R-CNN | epoque 1/10 | perte train 0.6079 | F1 valid 0.3679
Mask R-CNN | epoque 2/10 | perte train 0.3807 | F1 valid 0.5870
Mask R-CNN | epoque 3/10 | perte train 0.2934 | F1 valid 0.5542
Mask R-CNN | epoque 4/10 | perte train 0.2107 | F1 valid 0.6626
Mask R-CNN | epoque 5/10 | perte train 0.1827 | F1 valid 0.6914
Mask R-CNN | epoque 6/10 | perte train 0.1693 | F1 valid 0.6707
Mask R-CNN | epoque 7/10 | perte train 0.1580 | F1 valid 0.6727
Mask R-CNN | epoque 8/10 | perte train 0.1560 | F1 valid 0.6707
Mask R-CNN | epoque 9/10 | perte train 0.1551 | F1 valid 0.6790
Mask R-CNN | epoque 10/10 | perte train 0.1545 | F1 valid 0.6748
Mask R-CNN test (meilleure epoque 5) | precision=0.5761 | rappel=0.7067 | F1 masque=0.6347 (IoU >= 0.5)


In [8]:
# SAUVEGARDE : checkpoints, historiques et resultats d'evaluation de detection.
# Ultralytics a deja ecrit best.pt, last.pt et results.csv dans le dossier train de chaque modele.
for model_name, train_dir in yolo_train_dirs.items():
    if not (train_dir / "weights" / "best.pt").is_file():
        print(f"Checkpoint YOLO introuvable pour {model_name} ({train_dir}); verifier le journal.")

# Poids finaux Mask R-CNN (meilleure epoque) et historique.
mask_rcnn_checkpoint = mask_rcnn_train_dir / "mask_rcnn_brain_tumor.pth"
torch.save(
    {"model_state_dict": mask_rcnn_model.state_dict(), "class_names": CLASS_NAMES},
    mask_rcnn_checkpoint,
)
write_history_csv(mask_rcnn_history, mask_rcnn_train_dir / "training_history.csv")

# Resultats test de YOLO et Mask R-CNN dans un rapport commun (colonnes dans un ordre fixe).
# mAP : YOLO uniquement. Precision / rappel / F1 a IoU 0.5 : les deux, mais YOLO est evalue a son seuil de
# confiance optimal et Mask R-CNN a 0.5 : comparaison indicative.
detection_results = list(yolo_test_results)
detection_results.append(
    {
        "model": "Mask R-CNN",
        "best_epoch": mask_rcnn_test["best_epoch"],
        "precision_iou50": mask_rcnn_test["precision_iou50"],
        "recall_iou50": mask_rcnn_test["recall_iou50"],
        "f1_iou50": mask_rcnn_test["f1_iou50"],
    }
)
fieldnames = [
    "model", "best_epoch", "box_map50", "box_map50_95", "mask_map50", "mask_map50_95",
    "precision_iou50", "recall_iou50", "f1_iou50",
]
with (detection_eval_root / "test_metrics.csv").open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames, restval="")
    writer.writeheader()
    writer.writerows(detection_results)

print(f"Entrainement detection : {detection_train_root}")
print(f"Evaluation detection : {detection_eval_root / 'test_metrics.csv'}")

Entrainement detection : /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/outputs/detection/train
Evaluation detection : /Users/ikram/Desktop/github-project/etude_de_cas__paris_cite_M2/outputs/detection/evaluation/test_metrics.csv
